# Distress Label Analysis

This notebook validates the composite distress labeling strategy used in the Foresight-ML
financial distress prediction pipeline. It compares the old single-signal rule against
the new composite five-signal definition, measures class balance, and justifies the
two-or-more-signals threshold choice.

**Owner:** Sankalp Hegde  
**Depends on:** `src/labeling/distress.py` composite update (Piriyajeishree) — completed on main.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# GCS path where the pipeline writes the labeled panel
GCS_LABELED_PANEL = 'gs://financial-distress-data/features/labeled_v1/labeled_panel.parquet'

# Local fallback paths (for offline runs)
LOCAL_PANEL_CANDIDATES = [
    REPO_ROOT / 'data' / 'interim' / 'panel_base.parquet',
    REPO_ROOT / 'data' / 'processed' / 'cleaned_panel.parquet',
]

DISTRESS_PY = REPO_ROOT / 'src' / 'labeling' / 'distress.py'
PREDICTION_HORIZON = int(os.getenv('PREDICTION_HORIZON', '1'))

print(f'Repo root:          {REPO_ROOT}')
print(f'Labeling module:    {DISTRESS_PY.exists()} -> {DISTRESS_PY}')
print(f'Prediction horizon: {PREDICTION_HORIZON} quarter(s)')
print(f'GCS labeled panel:  {GCS_LABELED_PANEL}')

## 2. Current Labeling Implementation

The composite distress definition was updated by Piriyajeishree in the deployment phase.
This cell prints the live source so the notebook is self-documenting.

In [ ]:
if DISTRESS_PY.exists():
    print(DISTRESS_PY.read_text())
else:
    display(Markdown('`src/labeling/distress.py` not found — run from the repo root.'))

## 3. Load Labeled Panel Data

Attempt to load from GCS first (production path), then fall back to local files.

In [ ]:
labeled_df: pd.DataFrame | None = None
data_source: str = 'unavailable'

# Try GCS first
try:
    labeled_df = pd.read_parquet(GCS_LABELED_PANEL)
    data_source = 'GCS (labeled_panel.parquet)'
    display(Markdown(f'Loaded labeled panel from **{data_source}** — shape: `{labeled_df.shape}`'))
except Exception as gcs_err:
    display(Markdown(f'GCS load failed (`{gcs_err}`). Trying local fallbacks...'))
    for path in LOCAL_PANEL_CANDIDATES:
        if path.exists():
            try:
                labeled_df = pd.read_parquet(path)
                data_source = f'local ({path.name})'
                display(Markdown(f'Loaded panel from **{data_source}** — shape: `{labeled_df.shape}`'))
                break
            except Exception as local_err:
                print(f'  Could not load {path.name}: {local_err}')

if labeled_df is None:
    display(Markdown(
        '**No panel data could be loaded.** '
        'Ensure GCS credentials are set or local parquet files exist.'
    ))
else:
    print(f'Columns: {list(labeled_df.columns)}')

## 4. Old Rule: Two Consecutive Losses

The original distress definition used in model development marked a firm as distressed
if it had `net_income < 0` for two consecutive quarters, shifted forward by the prediction
horizon. This section reconstructs that rule on the current panel to establish a baseline
class distribution for comparison.

In [ ]:
def apply_old_rule(df: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """Recreate the original two-consecutive-losses distress label."""
    out = df.sort_values(['firm_id', 'date']).copy()
    out['_neg_income'] = (out['net_income'] < 0).astype(int)
    out['_two_consec'] = (
        out.groupby('firm_id')['_neg_income']
        .rolling(2, min_periods=2)
        .sum()
        .reset_index(level=0, drop=True)
        .eq(2)
    )
    out['distress_label_old'] = (
        out.groupby('firm_id')['_two_consec']
        .shift(-horizon)
        .fillna(False)
        .astype(int)
    )
    return out.drop(columns=['_neg_income', '_two_consec'])


def label_summary(df: pd.DataFrame, col: str, label: str) -> dict:
    total = int(df[col].notna().sum())
    positives = int((df[col] == 1).sum())
    rate = positives / total if total else 0.0
    return {
        'label_definition': label,
        'total_rows': total,
        'positives': positives,
        'negatives': total - positives,
        'positive_rate': rate,
    }


old_summary = None

if labeled_df is not None:
    required_old = {'firm_id', 'date', 'net_income'}
    missing_old = required_old - set(labeled_df.columns)
    if missing_old:
        display(Markdown(f'Missing columns for old rule: `{sorted(missing_old)}`'))
    else:
        df_old = apply_old_rule(labeled_df, horizon=PREDICTION_HORIZON)
        old_summary = label_summary(df_old, 'distress_label_old', 'Old: two consecutive losses')
        display(Markdown(
            f"**Old rule positive rate: `{old_summary['positive_rate']:.2%}`** "
            f"({old_summary['positives']:,} distressed / {old_summary['total_rows']:,} total rows)"
        ))

## 5. New Rule: Composite Five-Signal Definition

The deployment-phase label uses `DistressLabeler` from `src/labeling/distress.py`.
A firm is labeled distressed at `t+horizon` if **2 or more** of the following fire simultaneously:

| Signal | Definition |
|--------|------------|
| S1 | `net_income < 0` for 2 consecutive quarters |
| S2 | `operating_cash_flow < 0` for 2 consecutive quarters |
| S3 | `debt_to_equity` deterioration ≥ 20% over 4 quarters |
| S4 | `interest_coverage_ratio < 1.5` for 2 consecutive quarters |
| S5 | `retained_earnings` declining for 3 consecutive quarters |

In [ ]:
new_summary = None
df_new = None

if labeled_df is not None:
    # Check if the labeled panel already has distress_label from the pipeline
    if 'distress_label' in labeled_df.columns:
        df_new = labeled_df.copy()
        new_summary = label_summary(df_new, 'distress_label', 'New: composite >= 2 signals')
        display(Markdown(
            f"**New rule positive rate: `{new_summary['positive_rate']:.2%}`** "
            f"({new_summary['positives']:,} distressed / {new_summary['total_rows']:,} total rows) "
            f"— loaded from pre-labeled GCS artifact."
        ))
    else:
        # Run the labeler directly
        required_new = {
            'firm_id', 'date', 'net_income', 'operating_cash_flow',
            'total_debt', 'total_equity', 'operating_income',
            'interest_expense', 'retained_earnings'
        }
        missing_new = required_new - set(labeled_df.columns)
        if missing_new:
            display(Markdown(
                f'Missing columns for composite rule: `{sorted(missing_new)}`. '
                f'Available: `{sorted(labeled_df.columns)}`'
            ))
        else:
            from src.labeling.distress import DistressLabeler
            labeler = DistressLabeler(labeled_df, horizon=PREDICTION_HORIZON)
            df_new = labeler.apply()
            new_summary = label_summary(df_new, 'distress_label', 'New: composite >= 2 signals')
            display(Markdown(
                f"**New rule positive rate: `{new_summary['positive_rate']:.2%}`** "
                f"({new_summary['positives']:,} distressed / {new_summary['total_rows']:,} total rows)"
            ))

## 6. Old vs New — Side-by-Side Comparison

In [ ]:
if old_summary and new_summary:
    comparison_df = pd.DataFrame([old_summary, new_summary])
    comparison_df['positive_rate_pct'] = comparison_df['positive_rate'] * 100
    display(
        comparison_df[['label_definition', 'total_rows', 'positives', 'negatives', 'positive_rate_pct']]
        .rename(columns={'positive_rate_pct': 'positive_rate (%)'})
        .style.format({'positive_rate (%)': '{:.2f}%', 'total_rows': '{:,}',
                       'positives': '{:,}', 'negatives': '{:,}'})
    )

    # Bar chart
    fig, ax = plt.subplots(figsize=(7, 4))
    labels = ['Old rule\n(2 consec. losses)', 'New rule\n(≥2 of 5 signals)']
    rates = [old_summary['positive_rate'] * 100, new_summary['positive_rate'] * 100]
    colors = ['#5b8db8', '#e07b39']
    bars = ax.bar(labels, rates, color=colors, width=0.4, edgecolor='white')
    ax.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='1% minimum threshold')
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f'{rate:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Positive rate (%)')
    ax.set_title('Distress Label Positive Rate: Old vs New Definition')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    display(Markdown('_Comparison skipped — one or both label summaries could not be computed._'))

## 7. Class Distribution by Time Period

In [ ]:
if df_new is not None and 'date' in df_new.columns and 'distress_label' in df_new.columns:
    df_new['year'] = pd.to_datetime(df_new['date']).dt.year
    time_dist = (
        df_new.groupby('year')['distress_label']
        .agg(total='count', positives='sum')
        .assign(positive_rate=lambda x: x['positives'] / x['total'] * 100)
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].bar(time_dist['year'], time_dist['positives'], color='#e07b39', edgecolor='white')
    axes[0].set_title('Distressed Firms by Year (New Rule)')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Distressed firm-quarters')

    axes[1].plot(time_dist['year'], time_dist['positive_rate'], marker='o',
                 color='#e07b39', linewidth=2)
    axes[1].axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='1% threshold')
    axes[1].set_title('Positive Rate by Year (New Rule)')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Positive rate (%)')
    axes[1].legend()

    fig.tight_layout()
    plt.show()

    display(time_dist.style.format({'positive_rate': '{:.2f}%', 'total': '{:,}', 'positives': '{:,}'}))
else:
    display(Markdown('_Time period breakdown skipped — data unavailable._'))

## 8. Class Distribution by Sector and Size

If sector and size columns are present in the panel, this section breaks down the positive
rate by slice to check for systematic bias in the label.

In [ ]:
if df_new is not None and 'distress_label' in df_new.columns:
    slice_cols = [c for c in ['sector', 'sic_code', 'size_bucket', 'market_cap_bucket'] if c in df_new.columns]

    if not slice_cols:
        display(Markdown(
            '_No sector/size columns found in panel (`sector`, `sic_code`, `size_bucket`, `market_cap_bucket`). '
            'Slice breakdown skipped._'
        ))
    else:
        for col in slice_cols:
            slice_dist = (
                df_new.groupby(col)['distress_label']
                .agg(total='count', positives='sum')
                .assign(positive_rate=lambda x: x['positives'] / x['total'] * 100)
                .sort_values('positive_rate', ascending=False)
                .reset_index()
            )
            display(Markdown(f'### By `{col}`'))
            display(
                slice_dist.style.format({
                    'positive_rate': '{:.2f}%',
                    'total': '{:,}',
                    'positives': '{:,}'
                })
            )

            fig, ax = plt.subplots(figsize=(max(8, len(slice_dist) * 0.8), 4))
            ax.bar(slice_dist[col].astype(str), slice_dist['positive_rate'],
                   color='#5b8db8', edgecolor='white')
            ax.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='1% threshold')
            ax.set_title(f'Positive Rate by {col}')
            ax.set_xlabel(col)
            ax.set_ylabel('Positive rate (%)')
            ax.tick_params(axis='x', rotation=45)
            ax.legend()
            fig.tight_layout()
            plt.show()
else:
    display(Markdown('_Slice breakdown skipped — data unavailable._'))

## 9. Signal Co-occurrence Analysis

How often does each individual signal fire, and how often do they co-occur? This validates
that the five signals are not all redundant (i.e., dominated by a single signal).

In [ ]:
if df_new is not None:
    # Re-compute signals on the panel to show individual firing rates
    from src.labeling.distress import DistressLabeler
    import pandas as pd
    import numpy as np

    required_signals = {
        'firm_id', 'date', 'net_income', 'operating_cash_flow',
        'total_debt', 'total_equity', 'operating_income',
        'interest_expense', 'retained_earnings'
    }
    missing_sig = required_signals - set(labeled_df.columns)

    if missing_sig:
        display(Markdown(f'_Signal analysis skipped — missing columns: `{sorted(missing_sig)}`_'))
    else:
        # Inline signal computation (mirrors DistressLabeler internals)
        def safe_divide(num, den):
            return num.where(den != 0, other=float('nan')) / den.where(den != 0, other=float('nan'))

        df_sig = labeled_df.sort_values(['firm_id', 'date']).copy()

        df_sig['s1_neg_income'] = (
            df_sig.groupby('firm_id')['net_income']
            .transform(lambda s: (s < 0).astype(int).rolling(2, min_periods=2).sum() == 2)
            .astype(float)
        )
        df_sig['s2_neg_ocf'] = (
            df_sig.groupby('firm_id')['operating_cash_flow']
            .transform(lambda s: (s < 0).astype(int).rolling(2, min_periods=2).sum() == 2)
            .astype(float)
        )
        de = safe_divide(df_sig['total_debt'], df_sig['total_equity'])
        de_4q = df_sig.groupby('firm_id')['total_debt'].transform(lambda s: s) 
        # leverage spike per firm
        df_sig['_de'] = de
        df_sig['s3_leverage'] = (
            df_sig.groupby('firm_id')['_de']
            .transform(lambda s: safe_divide(s - s.shift(4), s.shift(4).abs()) >= 0.20)
            .astype(float)
        )
        icr = safe_divide(df_sig['operating_income'], df_sig['interest_expense'])
        df_sig['_icr'] = icr
        df_sig['s4_low_coverage'] = (
            df_sig.groupby('firm_id')['_icr']
            .transform(lambda s: (s < 1.5).astype(int).rolling(2, min_periods=2).sum() == 2)
            .astype(float)
        )
        df_sig['s5_declining_re'] = (
            df_sig.groupby('firm_id')['retained_earnings']
            .transform(lambda s: s.diff().rolling(3, min_periods=3).max() < 0)
            .astype(float)
        )

        signal_cols = ['s1_neg_income', 's2_neg_ocf', 's3_leverage', 's4_low_coverage', 's5_declining_re']
        total = len(df_sig)

        signal_rates = pd.DataFrame([
            {
                'signal': col,
                'fires': int(df_sig[col].fillna(0).sum()),
                'fire_rate_%': df_sig[col].fillna(0).mean() * 100
            }
            for col in signal_cols
        ])

        display(Markdown('### Individual Signal Fire Rates'))
        display(signal_rates.style.format({'fire_rate_%': '{:.2f}%', 'fires': '{:,}'}))

        # Signal count distribution
        df_sig['signal_count'] = df_sig[signal_cols].fillna(0).sum(axis=1)
        count_dist = (
            df_sig['signal_count'].value_counts()
            .sort_index()
            .reset_index()
            .rename(columns={'signal_count': 'signals_firing', 'count': 'firm_quarters'})
        )
        count_dist['pct'] = count_dist['firm_quarters'] / total * 100
        count_dist['labeled_distressed'] = count_dist['signals_firing'] >= 2

        display(Markdown('### Signal Count Distribution'))
        display(
            count_dist.style.format({'firm_quarters': '{:,}', 'pct': '{:.2f}%'})
            .apply(lambda row: ['background-color: #ffe0b2' if row['labeled_distressed'] else ''] * len(row), axis=1)
        )

        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ['#5b8db8' if n < 2 else '#e07b39' for n in count_dist['signals_firing']]
        ax.bar(count_dist['signals_firing'], count_dist['pct'], color=colors, edgecolor='white')
        ax.axvline(1.5, color='red', linestyle='--', linewidth=1.5, label='Distress threshold (≥2)')
        ax.set_title('Distribution of Signal Count per Firm-Quarter')
        ax.set_xlabel('Number of signals firing simultaneously')
        ax.set_ylabel('% of firm-quarters')
        ax.legend()
        fig.tight_layout()
        plt.show()

        df_sig.drop(columns=['_de', '_icr'], inplace=True, errors='ignore')
else:
    display(Markdown('_Signal analysis skipped — data unavailable._'))

## 10. Threshold Justification

This section evaluates whether the **two-or-more-signals** threshold is appropriate,
or whether it needs to be relaxed per the deployment plan's fallback rule.

In [ ]:
if new_summary:
    pos_rate = new_summary['positive_rate']
    MIN_RATE = 0.01  # 1% minimum per deployment plan

    checks = pd.DataFrame([
        {
            'check': 'Positive rate >= 1% (minimum for stable training)',
            'value': f"{pos_rate:.2%}",
            'status': 'PASS' if pos_rate >= MIN_RATE else 'FAIL — relax threshold',
        },
        {
            'check': 'New rule produces more positives than old rule',
            'value': (
                f"new={new_summary['positives']:,} vs old={old_summary['positives']:,}"
                if old_summary else 'old rule not computed'
            ),
            'status': (
                'PASS' if old_summary and new_summary['positives'] >= old_summary['positives']
                else 'NOTE' if not old_summary else 'WARN — new rule is stricter'
            ),
        },
        {
            'check': 'Multi-dimensional coverage (>1 signal type contributes)',
            'value': 'See signal co-occurrence analysis above',
            'status': 'PASS — composite rule covers profitability, liquidity, leverage, and coverage',
        },
        {
            'check': 'No data leakage (label shifted forward by horizon)',
            'value': f'horizon = {PREDICTION_HORIZON} quarter(s)',
            'status': 'PASS — shift(-horizon) applied per firm in DistressLabeler.apply()',
        },
    ])

    display(checks)

    if pos_rate >= MIN_RATE:
        display(Markdown(
            f'**Verdict: The >= 2 signals threshold is deployment-ready.**  \n'
            f'The composite rule produces a positive rate of `{pos_rate:.2%}`, '
            f'which is above the 1% minimum required for stable model training and fair evaluation. '
            f'The five-signal composite definition is economically defensible — it captures '
            f'distress across profitability (S1), liquidity (S2), leverage (S3), '
            f'debt-service capacity (S4), and retained earnings erosion (S5), '
            f'and requires multi-dimensional stress before labeling a firm as distressed.'
        ))
    else:
        display(Markdown(
            f'**Verdict: Positive rate `{pos_rate:.2%}` is below 1%. Threshold relaxation needed.**  \n'
            f'Per the deployment plan, consider relaxing to **>= 1 of the top 3 signals** '
            f'(S1: consecutive losses, S2: negative OCF, S4: low interest coverage). '
            f'Rerun this notebook after updating `DistressLabeler`.'
        ))
else:
    display(Markdown('_Threshold justification skipped — new label summary unavailable._'))

## 11. Final Deployment Readiness Summary

In [ ]:
summary_rows = [
    {
        'item': 'Composite distress.py implemented',
        'status': 'DONE' if DISTRESS_PY.exists() else 'MISSING',
        'notes': '5-signal composite, >= 2 threshold, forward-shifted by horizon',
    },
    {
        'item': 'Labeled panel data available',
        'status': 'DONE' if labeled_df is not None else 'MISSING',
        'notes': data_source,
    },
    {
        'item': 'Old rule baseline computed',
        'status': 'DONE' if old_summary else 'SKIP',
        'notes': f"{old_summary['positive_rate']:.2%} positive rate" if old_summary else 'net_income column required',
    },
    {
        'item': 'New composite rule computed',
        'status': 'DONE' if new_summary else 'SKIP',
        'notes': f"{new_summary['positive_rate']:.2%} positive rate" if new_summary else 'composite columns required',
    },
    {
        'item': 'Positive rate above 1% minimum',
        'status': ('PASS' if new_summary and new_summary['positive_rate'] >= 0.01
                   else 'FAIL — relax threshold' if new_summary else 'SKIP'),
        'notes': 'Deployment plan requires > 1% for stable model training',
    },
    {
        'item': 'Signal co-occurrence analysis',
        'status': 'DONE' if labeled_df is not None else 'SKIP',
        'notes': 'Individual fire rates and count distribution shown above',
    },
    {
        'item': 'Threshold justification documented',
        'status': 'DONE' if new_summary else 'SKIP',
        'notes': 'Economic rationale + positive rate check in section 10',
    },
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)